# Relatório do pipeline DCJ (SIM x SIH)
Este notebook **só lê e mostra** resultados. Gere-os antes no terminal:

```
python -m dcj
```

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from dcj.configuracao import carregar_configuracao

PASTA = Path(r"C:\Users\isabe\Downloads\projeto_dcj\dados_rs_drp\resultados")

def ler(nome: str, **argumentos) -> pd.DataFrame:
    return pd.read_csv(PASTA / f"{nome}.csv", **argumentos)

print("Resultados em:", PASTA)

In [ ]:
import pandas as pd
from dcj.fonte_pysus import obter_obitos_do_pysus

# 1. O que o PySUS entrega AGORA?
print("--- Baixando do PySUS novamente ---")
novo = obter_obitos_do_pysus("RS", 2019)
print("Tamanho do download novo:", len(novo))
print("Top 5 prefixos de UF:", novo["CODMUNRES"].astype(str).str[:2].value_counts().head(5).to_dict())

# 2. O que tem no arquivo em disco?
print("\n--- Lendo o arquivo em disco ---")
disco = pd.read_parquet(r"C:\Users\isabe\Downloads\projeto_dcj\dados_rs_drp\sim\sim_rs_2019.parquet")
print("Tamanho do arquivo em disco:", len(disco))
print("Top 5 prefixos de UF:", disco["CODMUNRES"].astype(str).str[:2].value_counts().head(5).to_dict())

## 1. Funil de ingestão dos óbitos
Quanto do que foi baixado é residente da UF e não duplicado.

In [ ]:
funil = ler("funil_obitos")
display(funil)

figura, eixo = plt.subplots(figsize=(9, 3.6))
eixo.barh(funil["etapa"][::-1], funil["percentual_do_bruto"][::-1], color="#4682b4", edgecolor="black")
for posicao, (percentual, total) in enumerate(zip(funil["percentual_do_bruto"][::-1], funil["registros"][::-1])):
    eixo.text(percentual + 1, posicao, f"{percentual:.1f}%  (N={total:,})", va="center")
eixo.set_xlim(0, 125)
eixo.set_xlabel("% do total bruto")
plt.tight_layout()
plt.show()

## 2. Pareamento e cobertura por ano
Valores longe de 1,0 em `*_sobre_mediana` indicam ano com cobertura anômala.

In [ ]:
display(ler("pareamento_resumo", index_col=0))
display(ler("cobertura_por_ano", index_col=0))

## 3. Escore de suspeição
Com poucos pares as taxas são instáveis: veja o intervalo de confiança e a taxa por 100 internações.

In [ ]:
display(ler("escore_por_periodo", index_col=0))
display(ler("escore_taxa_normalizada"))
display(ler("escore_sensibilidade_desempate"))
display(ler("escore_sensibilidade_faixa_etaria"))

In [ ]:
colunas = ["data_internacao", "data_obito", "dias_ate_obito", "idade_anos", "diagnostico_principal",
           "causa_basica", "municipio", "escore_dcj", "periodo"]
casos = ler("casos_sinalizados")
display(casos[[coluna for coluna in colunas if coluna in casos.columns]])

## 4. Validação
`observado` é o número de pares do pipeline; `media_ao_acaso` e `percentil_95_ao_acaso` são o que o acaso produz nos mesmos blocos. Se `p_valor_unilateral` da última linha não for pequeno, os pares finais não se distinguem do acaso.

In [ ]:
display(ler("permutacao"))
display(ler("concordancia", index_col=0))

## 5. Referência: Boletim Epidemiológico DCJ (SVS/MS, vol. 53, n. 39, 2022)
A faixa 30-64 do escore exclui cerca de 44% das notificações suspeitas (65 anos ou mais; p. 21). A tabela de sensibilidade (seção 3) mostra quantos sinalizados mudam com a faixa 55-74 (p. 18). Óbitos A81.0 abaixo de 1 a 2 por milhão por ano (p. 12) podem indicar subnotificação ou cobertura incompleta do SIM.

In [ ]:
arquivo = PASTA / "obitos_a810_vs_esperado.csv"
if arquivo.exists():
    display(pd.read_csv(arquivo, index_col=0))
else:
    print("Rode `python -m dcj --etapa validacao --populacao N` para gerar esta tabela.")

## 6. Cruzamento exploratório com o SINAN
Chave fraca (ano de nascimento, sexo, município): use para revisão manual, não para proporções.

In [ ]:
arquivo_sinan = PASTA / "sinan_resumo.csv"
if arquivo_sinan.exists():
    display(pd.read_csv(arquivo_sinan, index_col=0))
else:
    print("Sem SINAN: coloque sinan/sinan_rs_a81x_todos_anos.parquet e rode --etapa sinan.")

In [ ]:
for nome in ("permutacao_sinan", "sinan_perfil_individuos"):
    arquivo = PASTA / f"{nome}.csv"
    if arquivo.exists():
        display(pd.read_csv(arquivo))

In [ ]:
import numpy as np
import pandas as pd
import glob

# 1. Carregar os dados (SIM e SINAN)
arquivos_sim = glob.glob(r"C:\Users\isabe\Downloads\projeto_dcj\dados_rs_drp\sim\sim_rs_*.parquet")
df_sim_total = pd.concat([pd.read_parquet(f) for f in arquivos_sim], ignore_index=True)

arquivos_sinan = glob.glob(r"C:\Users\isabe\Downloads\projeto_dcj\dados_rs_drp\sinan\sinan_rs_*.parquet")
df_sinan = pd.concat([pd.read_parquet(f) for f in arquivos_sinan], ignore_index=True)

# Filtro do SINAN para RS
col_mun = "ID_MN_RESI" if "ID_MN_RESI" in df_sinan.columns else "MUNICIPIO"
df_sinan[col_mun] = df_sinan[col_mun].astype("string").str.replace(r"\.0$", "", regex=True)
df_sinan = df_sinan[df_sinan[col_mun].str.startswith("43", na=False)]

for c in ["CLASSI_FIN", "EVOLUCAO"]:
    if c in df_sinan.columns:
        df_sinan[c] = df_sinan[c].astype("string").str.replace(r"\.0$", "", regex=True)

# 2. Funções do revisor
JANELA_OBITO_DIAS = 30
JANELA_SINTOMA    = (-30, 365)

def _dmy(s):
    if pd.api.types.is_datetime64_any_dtype(s): return s
    s = s.astype("string").str.replace(r"\.0$", "", regex=True).str.zfill(8)
    return pd.to_datetime(s, format="%d%m%Y", errors="coerce")

def _mun6(s): return s.astype("string").str.replace(r"\.0$", "", regex=True).str.strip().str.zfill(6).str[:6]
def _sexo(s): return s.astype("string").str.strip().str.upper().replace({"1": "M", "2": "F"})

def preparar_sinan(df):
    s = df.reset_index(drop=True).copy()
    s["SID"] = s.index
    col_sexo = "CS_SEXO" if "CS_SEXO" in s.columns else "SEXO"
    col_ano = "ANO_NASC_LIMPO" if "ANO_NASC_LIMPO" in s.columns else ("ANO_NASC" if "ANO_NASC" in s.columns else "ANO_NASCIMENTO")
    col_dt_sin = "DT_SIN_PRI" if "DT_SIN_PRI" in s.columns else "DATA_PRIMEIROS_SINTOMAS"
    col_dt_obito = "DT_OBITO" if "DT_OBITO" in s.columns else "DATA_OBITO"
    
    s["MUN6"] = _mun6(s[col_mun])
    s["SEXO"] = _sexo(s[col_sexo])
    s["ANO_N"] = pd.to_numeric(s[col_ano], errors="coerce").astype("Int64")
    s["DT_SIN"] = pd.to_datetime(s[col_dt_sin], errors="coerce")
    s["DT_OBITO_SN"] = pd.to_datetime(s[col_dt_obito], errors="coerce") if col_dt_obito in s.columns else pd.NaT
    return s

def preparar_sim(df):
    d = df.drop_duplicates().reset_index(drop=True).copy()
    d["OID"] = d.index
    d["MUN6"] = _mun6(d["CODMUNRES"])
    d = d[d["MUN6"].str.startswith("43")].copy()
    d["SEXO"] = _sexo(d["SEXO"])
    d["DTN"] = _dmy(d["DTNASC"])
    d["DTO"] = _dmy(d["DTOBITO"])
    d["ANO_N"] = d["DTN"].dt.year.astype("Int64")
    d["E_A81"] = d["CAUSABAS"].astype("string").str.upper().str.match(r"^A81[09]").fillna(False).astype(bool)
    return d

def cruzar(sn, sim):
    m = sim.merge(sn[["SID", "MUN6", "SEXO", "ANO_N", "DT_SIN", "DT_OBITO_SN"]], on=["MUN6", "SEXO", "ANO_N"], how="inner")
    m["DIAS_SINTOMA_OBITO"] = (m["DTO"] - m["DT_SIN"]).dt.days
    lo, hi = JANELA_SINTOMA
    m = m[m["DIAS_SINTOMA_OBITO"].between(lo, hi)].copy()
    m["DELTA_OBITO"] = (m["DTO"] - m["DT_OBITO_SN"].dt.normalize()).dt.days
    ind = "indeterminado (sem DT_OBITO no SINAN)"
    m["CHAVE_OBITO"] = np.select(
        [m["DELTA_OBITO"].isna(), m["DELTA_OBITO"].abs() <= JANELA_OBITO_DIAS],
        [ind, "forte"], default="incompatível")
    return m

# 3. Executa tudo de uma vez
sn  = preparar_sinan(df_sinan)
sim = preparar_sim(df_sim_total)
m = cruzar(sn, sim)

# 4. Auditoria de mascaramento (chave forte)
mascarados_fortes = m[(m["CHAVE_OBITO"] == "forte") & (~m["E_A81"])]
cols_auditoria = ["SID", "MUN6", "SEXO", "ANO_N", "DT_SIN", "DT_OBITO_SN", "DTO", "CAUSABAS"]
tabela_auditoria = mascarados_fortes[cols_auditoria].sort_values("DTO")

tabela_auditoria.to_csv(PASTA / "auditoria_mascaramento_confirmado.csv", index=False)
print(f"Total de casos de mascaramento CONFIRMADOS (chave forte): {len(tabela_auditoria)}")
display(tabela_auditoria)

In [ ]:
# Verifica o nome exato das colunas
print(df_sinan.columns.tolist())

# Se a coluna for DT_OBITO, veja como os valores estão armazenados (antes de tentar converter)
if "DT_OBITO" in df_sinan.columns:
    print("Exemplo de DT_OBITO:", df_sinan["DT_OBITO"].dropna().head(3).tolist())
elif "DATA_OBITO" in df_sinan.columns:
    print("Exemplo de DATA_OBITO:", df_sinan["DATA_OBITO"].dropna().head(3).tolist())
else:
    print("Nenhuma coluna de data de óbito encontrada!")

In [ ]:
from dcj.captura_recaptura import DuasListas, montar_tabela_de_sensibilidade

# Dados extraídos do seu pipeline (ajuste se necessário)
listas_todas = DuasListas(total_da_primeira=45, total_da_segunda=44, em_ambas=10)
listas_confirmadas = DuasListas(total_da_primeira=28, total_da_segunda=44, em_ambas=8)

cenarios = {
    "SINAN todas (45)": listas_todas,
    "SINAN confirmadas (28)": listas_confirmadas,
}

tabela_sensibilidade = montar_tabela_de_sensibilidade(cenarios)
display(tabela_sensibilidade)
tabela_sensibilidade.to_csv(PASTA / "captura_recaptura_sensibilidade.csv", index=False)